In [0]:
%sql
-- CELL 1: State/Area Dimension
CREATE OR REPLACE TABLE cms.gold.dim_state
COMMENT 'Prescriber state / area'
AS
SELECT state_code, state_name, area_type
FROM cms.silver.ref_state;

-- A dimension table describes the who, what, and where of each row. The fact table records the how much. dim_state: OH | Ohio | State

In [0]:
%sql
-- CELL 2: Rest of the Dimensions
CREATE OR REPLACE TABLE cms.gold.dim_specialty
COMMENT 'Prescriber specialty and group'
AS
SELECT specialty, specialty_group
FROM cms.silver.ref_specialty;
CREATE OR REPLACE TABLE cms.gold.dim_drug
COMMENT 'Drug = brand name + generic name pair. drug_key is a deterministic hash of the pair.'
AS
SELECT
  xxhash64(brand_name, generic_name) AS drug_key,
  brand_name,
  generic_name
FROM (
  SELECT DISTINCT brand_name, generic_name
  FROM cms.silver.partd_prescriber_drug
);
CREATE OR REPLACE TABLE cms.gold.dim_prescriber
COMMENT 'One row per prescriber NPI'
AS
SELECT DISTINCT
  prescriber_npi,
  CASE WHEN is_organization THEN prescriber_last_or_org_name
       ELSE concat_ws(', ', prescriber_last_or_org_name, prescriber_first_name)
  END                         AS prescriber_name,
  is_organization,
  prescriber_city
FROM cms.silver.partd_prescriber_drug;

-- dim_prescriber: 1003000142 | Khalil, Rashid | Toledo
-- dim_drug: drug_key | Baclofen | Baclofen
-- dim_specialty: Anesthesiology | Other
-- In short, dimensions are what you click on to filter and slice; the fact table is what gets added up.

In [0]:
%sql
-- CELL 3: Fact Table
CREATE OR REPLACE TABLE cms.gold.fact_prescriber_drug
COMMENT 'Grain: one row per data_year x prescriber x drug. Beneficiaries are NULL when CMS suppressed them (1-10).'
AS
SELECT
  data_year,
  prescriber_npi,
  xxhash64(brand_name, generic_name) AS drug_key,
  state_code,
  specialty,
  total_claims,
  total_30day_fills,
  total_day_supply,
  total_drug_cost,
  total_beneficiaries,
  beneficiaries_suppressed
FROM cms.silver.partd_prescriber_drug;

-- fact: 2024 | 1003000142 | drug_key | OH | Anesthesiology | 47 | 333.35 (just the numbers and the keys)
-- The fact table can compute the key itself, without joining to dim_drug. A star schema keeps each dimension flat, exactly one join away from the fact table. 
-- 65+ columns were left out on purpose and stay in silver, because they’re too heavily hidden to be reliable.

In [0]:
## CELL 4: Dimension Check

fact = "cms.gold.fact_prescriber_drug"
dims = {
    "dim_state":      "state_code",
    "dim_specialty":  "specialty",
    "dim_drug":       "drug_key",
    "dim_prescriber": "prescriber_npi",
}

for dim, key in dims.items():
    t = spark.table(f"cms.gold.{dim}")
    n, n_keys = t.count(), t.select(key).distinct().count()
    assert n == n_keys, f"{dim}: {n - n_keys} duplicate {key} values"

    orphans = spark.sql(f"""
        SELECT COUNT(*) AS n
        FROM {fact} AS f
        LEFT ANTI JOIN cms.gold.{dim} AS d
          ON f.{key} = d.{key}
    """).first().n
    assert orphans == 0, f"{orphans:,} fact rows have no match in {dim}"

    print(f"{dim:15} {n:>10,} rows | unique key | no orphan fact rows")

## Cell 6 checks the two rules every star schema depends on:

## Each dimension has exactly one row per key. Otherwise a fact row would match two dimension rows, and its cost would be counted twice in Power BI.
## Every fact row finds its dimension row. Otherwise it shows up as “(Blank).”    

In [0]:
# CELL 5: Reconcile with Silver

g = spark.sql(f"""
SELECT COUNT(*)                                            AS n,
       COUNT(DISTINCT data_year, prescriber_npi, drug_key) AS n_grain,
       SUM(total_drug_cost)                                AS cost
FROM {fact}
""").first()

s = spark.sql("""
SELECT COUNT(*) AS n, SUM(total_drug_cost) AS cost
FROM cms.silver.partd_prescriber_drug
""").first()

assert g.n == s.n,       f"Row count: fact {g.n:,} vs silver {s.n:,}"
assert g.n == g.n_grain, f"Grain broken: {g.n - g.n_grain:,} duplicate year/prescriber/drug rows"
assert g.cost == s.cost, f"Cost doesn't reconcile: fact {g.cost} vs silver {s.cost}"

print(f"Fact: {g.n:,} rows, grain unique, total cost ${g.cost:,.2f} reconciles to silver")

## The total cost in gold must match silver to the cent, and the stated grain must actually hold.